# Phase 3 — RAG layer: indexing + retrieval

This notebook does NOT need a GPU — embedding with `all-MiniLM-L6-v2` and querying a local Qdrant collection both run fine on CPU. Runtime > Change runtime type > CPU is fine here, unlike `02_finetune.ipynb`.

Builds a searchable vector index over the full report corpus (not a train/val/test split — this simulates a real historical case archive), then tests that retrieval actually surfaces sensible similar reports before any LLM generation gets wired on top.

## 1. Mount Drive and pull the repo

Same Drive-persisted setup as the other notebooks.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os

from google.colab import userdata

GH_TOKEN = userdata.get("GH_TOKEN")
REPO_DIR = "/content/drive/MyDrive/Clinical_Report_Assistant"
REPO_URL = f"https://{GH_TOKEN}@github.com/ozgurberat/clinical-report-assistant.git"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already exists in Drive — pulling latest instead of re-cloning.")
    %cd {REPO_DIR}
    !git checkout -- notebooks/*.ipynb
    !git pull
else:
    !git clone {REPO_URL} "{REPO_DIR}"
    %cd {REPO_DIR}

## 2. Install the RAG dependencies

`sentence-transformers` (the embedding model) and `qdrant-client` (the local vector store) — neither needs a GPU.

In [ ]:
!pip install -q sentence-transformers qdrant-client pyyaml

## 3. Build the index

Embeds every report in `data/processed/reports.jsonl` and writes a local Qdrant collection to `data/processed/qdrant_index/`. Takes a few minutes on CPU for ~3,400 reports — this only needs to run once (or again if the underlying corpus changes); the index persists in Drive afterward.

In [ ]:
!python -m src.rag.build_index --processed data/processed

## 4. Test retrieval — does this actually surface similar reports?

Before building anything on top of this, read the actual matches by eye. A good sign: the top hits should share real clinical similarity with the query (similar findings/diagnosis), not just superficial word overlap — that's the whole point of embedding-based search over keyword search.

In [ ]:
!python -m src.rag.retrieve --query "mild cardiomegaly, clear lungs" --top-k 5

In [ ]:
!python -m src.rag.retrieve --query "pneumothorax with chest tube in place" --top-k 5

## Next

Read through both sets of retrieved matches. Do the top hits genuinely look clinically similar to the query, not just coincidentally sharing a word or two? If so, retrieval is working and we can move on to wiring an actual answer-generation step on top (`src/rag/qa.py`): retrieve the relevant past reports for a user's question, build a prompt embedding that evidence as context, and generate a grounded answer with the plain base Qwen3-4B model — no adapter attached, per the earlier adapter-vs-base experiment. If the matches look off (e.g. consistently irrelevant, or dominated by one giant cluster), bring back specific examples and we'll dig into why before going further.